# 01 · Build the structured dataset

Merges, for one financial year:
- NAO **auditor reports** for NHS trusts (opinion on financial statements, value-for-money conclusion, additional reporting powers, link to the report)
- the **organisation code lookup** (trust name → org code)
- trust **expenses** (expected vs actual cost)

Run once per year (2017-18 to 2021-22) and combine the outputs into `data/finaldatasetdissertation.xlsx`.

In [ ]:
import pandas as pd

# Source files go in data/raw/ (see README)
YEAR = "2018-2019"
RAW = "../data/raw"

df_audit = pd.read_excel(f"{RAW}/{YEAR}/2018auditreports.xlsx", header=1)
df_trust_to_orgcode = pd.read_excel(f"{RAW}/Organisation Description.xlsx", header=1)
df_expenses = pd.read_excel(f"{RAW}/{YEAR}/expenses.xlsx")

In [ ]:
# Attach the org code to each audit report
df_audit = df_audit.merge(
    df_trust_to_orgcode[['Org Name', 'Org Code']],
    left_on='BodyName', right_on='Org Name',
    how='left'
).drop(columns='Org Name')

In [ ]:
# Columns from the audit reports to add to the expenses data
columns_to_add = ['OpinionOnFinancialStatements', 'VfmArrangementsConclusion',
                  'AdditionalReportingPowers', 'AuditorReportOrLetterLink']

df_result = pd.merge(
    df_expenses,
    df_audit[['Org Code'] + columns_to_add],
    left_on='Row Labels',
    right_on='Org Code',
    how='inner'
).drop(columns=['Org Code'])  # duplicate of 'Row Labels'

df_result.head()

In [ ]:
df_result.to_excel(f"../data/WrittenDataset{YEAR}.xlsx", index=False)